# Phase 6: Metrics Agent (Subscriber + Publisher)

This notebook subscribes to spectator events and publishes final KPI metrics at halftime end.

In [1]:
# Cell 2 purpose: Import modules and ensure src is available from notebooks/.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.metrics import (
    MetricsAggregatorState,
    enforce_final_scenario_policies,
    finalize_kpi_payload,
    record_spectator_event,
)
from simulated_city.topic_schema import topic_kpi_metrics, topic_spectator_events

In [5]:
# Cell 3 purpose: Load config, create metrics aggregator, and connect MQTT client.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

enforce_final_scenario_policies(
    missed_kickoff_timestamp_s=900,
    external_disruptions_enabled=False,
    group_coordination_share=0.2,
)
state = MetricsAggregatorState(halftime_duration_s=900)

spectator_topic = topic_spectator_events()
kpi_topic = topic_kpi_metrics()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='metrics-agent')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topic: {spectator_topic}')
print(f'Publish topic: {kpi_topic}')

Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/events/spectator
Publish topic: stadium/a4/halftime/metrics/kpi


In [6]:
# Cell 4 purpose: Subscribe to spectator events and feed metrics aggregator.
received_events = []
published_kpi_payloads = []

def _on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    received_events.append(payload)
    record_spectator_event(state, payload)

mqtt_client.on_message = _on_message
mqtt_client.subscribe(spectator_topic, qos=1)
print('Subscription started. Waiting for incoming spectator events...')

Subscription started. Waiting for incoming spectator events...


In [4]:
# Cell 5 purpose: Run short loop, publish final KPI snapshot, and disconnect.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

run_id = received_events[-1]['run_id'] if received_events else 'a4-no-events'
final_payload = finalize_kpi_payload(state=state, run_id=run_id, timestamp_s=900)
ok = mqtt.publish_json_checked(mqtt_client, kpi_topic, final_payload, qos=1)
if ok:
    published_kpi_payloads.append(final_payload)

print(f'Received spectator events: {len(received_events)}')
print(f'Published KPI payloads: {len(published_kpi_payloads)}')
if published_kpi_payloads:
    print('Final KPI payload:')
    print(published_kpi_payloads[-1])

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

Received spectator events: 181
Published KPI payloads: 1
Final KPI payload:
{'schema_version': '1.0', 'run_id': 'a4-run-eb25aa71', 'timestamp_s': 900, 'average_wait_s': 106.94254143646408, 'wait_percentiles_s': {'P01': 0.0, 'P02': 0.0, 'P03': 0.0, 'P04': 0.0, 'P05': 0.0, 'P06': 0.0, 'P07': 0.0, 'P08': 0.0, 'P09': 0.0, 'P10': 0.0, 'P11': 0.0, 'P12': 0.6, 'P13': 1.6909090909090898, 'P14': 1.7999999999999998, 'P15': 2.072727272727272, 'P16': 2.563636363636364, 'P17': 3.109090909090908, 'P18': 4.745454545454546, 'P19': 5.236363636363635, 'P20': 6.6, 'P21': 7.41818181818182, 'P22': 9.218181818181815, 'P23': 12.0, 'P24': 13.690909090909091, 'P25': 14.945454545454538, 'P26': 17.618181818181814, 'P27': 22.581818181818182, 'P28': 30.163636363636368, 'P29': 31.799999999999997, 'P30': 34.418181818181814, 'P31': 36.0, 'P32': 37.90909090909092, 'P33': 41.61818181818182, 'P34': 43.199999999999996, 'P35': 47.67272727272727, 'P36': 51.76363636363636, 'P37': 54.0, 'P38': 57.2181818181818, 'P39': 62.290

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Disconnected from MQTT broker.
